## Similarity Scores - Cosine, Euclidean, SSIM

In [ ]:
import nibabel as nib
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from skimage.metrics import structural_similarity as ssim

avg_img = nib.load("avg_img.nii").get_fdata()
subject_img = nib.load("subjects_img.nii").get_fdata()

assert avg_img.shape == subject_img.shape[:3], "Shape mismatch between average and subject image"

avg_flat = avg_img.flatten()

T = subject_img.shape[3]
subject_2d = subject_img.reshape(-1, T)

mask = avg_flat != 0
avg_masked = avg_flat[mask]
subject_masked = subject_2d[mask, :]  

similarities = []
for t in range(T):
    sim = cosine_similarity(
        avg_masked.reshape(1, -1), 
        subject_masked[:, t].reshape(1, -1)
    )[0, 0]
    similarities.append(sim)

similarities = np.array(similarities)

print("Cosine similarity for each timepoint:")
print(similarities)
print("Mean similarity:", similarities.mean())

np.save("cosine_similarities.npy", similarities)


avg_flat = avg_img.flatten()  
subjects_flat = subject_img.reshape(-1, subject_img.shape[3]) 

euclidean_distances = np.linalg.norm(subjects_flat - avg_flat[:, np.newaxis], axis=0) 

euclidean_similarities = 1 / (1 + euclidean_distances)

np.save("euclidean_distances.npy", euclidean_distances)
np.save("euclidean_similarities.npy", euclidean_similarities)


n_volumes = subject_img.shape[3]
ssim_scores = []

for i in range(n_volumes):
    subj_img = subject_img[..., i]

    # along z-axis
    slice_ssims = []
    for z in range(avg_img.shape[2]):
        ref = avg_img[:, :, z]
        cmp = subj_img[:, :, z]

        if np.all(ref == 0) and np.all(cmp == 0):
            continue

        score, _ = ssim(
            ref,
            cmp,
            data_range=ref.max() - ref.min() + 1e-8,  
            full=True
        )
        slice_ssims.append(score)

    ssim_scores.append(np.mean(slice_ssims))

ssim_scores = np.array(ssim_scores)
np.save("ssim_scores_subjects.npy", ssim_scores)

print("SSIM computed for all volumes. Mean:", ssim_scores.mean())

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

similarity_scores = np.load("ssim_scores_subjects.npy") 
meta_data_df =  pd.read_csv("meta_data.csv")

score_columns = ['score1','score2','score3' ]

results = []

for score in score_columns:
    scores = meta_data_df[score].values
    # Drop NaNs pairwise
    mask = ~np.isnan(scores) & ~np.isnan(similarity_scores)
    if mask.sum() > 2:
        r, p = pearsonr(scores[mask], similarity_scores[mask])
        results.append({"score": score, "r": r, "p": p})
    else:
        results.append({"score": score, "r": np.nan, "p": np.nan})

results_df = pd.DataFrame(results)

results_df

valid = results_df["p"].notna()
pvals = results_df.loc[valid, "p"].values
print(set(valid.values))

from statsmodels.stats.multitest import multipletests
rej_bonf, pval_bonf, _, _ = multipletests(pvals, alpha=0.05, method='bonferroni')

results_df.loc[valid, "bonf_pval"] = pval_bonf
results_df.loc[valid, "bonf_significant"] = rej_bonf
results_df.to_csv("healthy_ssim.csv")

results_df 